# Install crucial libraries

In [ ]:
!pip install langchain-community
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5

In [ ]:
!pip install langchain
!pip install pypdf
!pip install langchain-chroma

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.2/304.2 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/19.3 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.8/65.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.7/55.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.2/196.2 kB 16.4 MB/s eta 0:

# Imports

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
import pandas as pd
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer
from tqdm.notebook import tqdm
import pandas as pd
import matplotlib.pyplot as plt
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores.utils import DistanceStrategy
from typing import Optional, List, Tuple
from langchain_chroma import Chroma
from transformers import pipeline
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
pd.set_option("display.max_colwidth", None)

# **utils**

In [ ]:
def split_documents(chunk_size: int, chunk_overlap: int, knowledge_base: List, tokenizer_name: str) -> List:
    """
    Split documents into chunks of maximum size `chunk_size` tokens and return a list of documents.
    """
    text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
        AutoTokenizer.from_pretrained(tokenizer_name),
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )

    docs_processed = text_splitter.split_documents(knowledge_base)

    return docs_processed


## **Part 0: Choose and Test Your Topic Without a Knowledge Base**

Before you load any external documents, you must **verify that your chosen topic needs a knowledge base** to improve answers. This ensures your RAG system solves a real gap in the model’s knowledge.

###  **Steps:**

1. **Choose a Topic (Tentative)**

   * Pick a topic from 2024 or 2025 that you think is recent or under-documented.
   * Example topics:

     * A political decision (e.g., "European Union climate laws in 2024")
     * A cultural trend (e.g., "Music trends in early 2025")

2. **Formulate Question**

   * Write down one factual, clear question about the topic.
   * Aim for question that require up-to-date or specific knowledge.

3. **Query the Model Directly**

   * Use your LLM pipeline (without RAG) to ask this question.
   * Collect the model’s answer and evaluate their quality:

     * Are the answers incomplete?
     * Are they outdated?
     * Are they confident but wrong?
     * Do they say *"I don’t know"*?

---

Why This Matters:

This step ensures your RAG project is solving a **real information gap**, not just repeating what the model already knows.


In [ ]:
question_part0="When did Pope Francis died?"

In [ ]:
READER_MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(READER_MODEL_NAME, quantization_config=bnb_config)
tokenizer = AutoTokenizer.from_pretrained(READER_MODEL_NAME)

In [ ]:
READER_LLM = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    temperature=0.2,
    repetition_penalty=1.1,
    return_full_text=False,
    max_new_tokens=500,
)

Device set to use cuda:0


In [ ]:
answer_part0=READER_LLM(question_part0)[0]["generated_text"]

In [ ]:
answer_part0

"\n\n### Answer:Pope Francis, born as Jorge Mario Bergoglio on December 17, 1936, passed away due to congestive heart failure. He was the head of the Catholic Church and sovereign of Vatican City from March 28, 2013, until his death. His papacy lasted for over eight years during which he made significant changes in various aspects including social issues like poverty and climate change. The news about his passing spread worldwide with many mourning his leadership and teachings. However, it's important to note that while there are speculations around when exactly a person might die based on their health condition or age, no one can predict an individual’s exact time of demise accurately."

it is very confident of the completly wrong answer

# **Part 1: Load a Custom PDF Knowledge Base**

Find blog posts or wikipedia page with your topic and save information about it to a PDF file, and load it using `PyPDFLoader`. You may use other loaders not only pdf, but pdf loader is exactly the same as we used during lab.

- Find informative content on your topic (Wikipedia page, blog post, article, etc.)
- Save the page as a PDF file (you can use your browser’s print-to-PDF feature)

In [ ]:
# Define the path to the PDF file containing the knowledge base
file_path = "/content/2025 papal conclave - Wikipedia.pdf"

# Create a PyPDFLoader object to handle loading of the PDF
loader = PyPDFLoader(file_path)

# Load the contents of the PDF into a variable for further processing
RAW_KNOWLEDGE_BASE = loader.load()


# **Part 2: Repeat the Lab with Your Own Knowledge Base + RAG Tuning**

## **Goal:**

Practice building a **RAG pipeline** and explore how **chunk size** and **chunk overlap** affect the quality of LLM answers to different questions.

---

## **What You Need to Do:**

1. **Repeat the Lab Using Your PDF Knowledge Base**

   * Use the PDF file you selected and loaded in Part 1.

2. **Create 3 Different Questions**

   * Design **three meaningful, specific questions** based on your topic.
   * Each question must be clearly related to the content of your PDF.

3. **Run RAG for Each Question with 3 Different Settings:**
   For each question:

   * Run the RAG pipeline **three times** using different settings for:

     * `chunk_size` (e.g., 100, 300, 500)
     * `chunk_overlap` (e.g., 0, 20, 50, 100)
   * This means you will run a total of **9 tests** (3 questions × 3 settings each).


4. **Answer Each Question Using an LLM**

   * Use the loaded chunks and a retriever to find relevant parts.
   * Pass the retrieved context to the LLM and generate an answer.
   * You can use similar tools as we used in the Lab

5. **Explain Your Results**
   For each of the 3 questions:

   * Write a short **description of the question** and **why you chose it**.
   * **Compare the answers** you got using different settings.
   * Reflect on:

     * How answer quality changed with different `chunk_size` and `chunk_overlap`
     * Which setting gave the most useful or accurate result
     * Why you think it performed better/worse

---

## **Deliverables:**

* Python code used for RAG pipeline (with different chunking settings)
* PDF file from Part 1
* A JSON file named rag_report_last_name_name_id.json containing your results:

  * 3 questions with explanations
  * Generated answers for each setting
  * Comparison and reflection on the results

---


In [ ]:
questions=["Who is the new Pope in 2025?",
           "Which cardinals did not participate in the 2025 Conclave?",
           "What are the post-conclave events?"]

In [ ]:
reasons=["i want to fact check",
         "requires to identify a list of individuals",
         "i a summary"]

In [ ]:
chunk_sizes=[100,600,600]
chunk_overlaps=[20,200,150]

In [ ]:
EMBEDDING_MODEL_NAME = "thenlper/gte-small"

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    multi_process=True,
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}  # Set `True` for cosine similarity
)

In [ ]:
all_vector_stores=[]
for i in range(len(chunk_sizes)):
  docs_processed = split_documents(
      chunk_size=chunk_sizes[i],
      chunk_overlap=chunk_overlaps[i],
      knowledge_base=RAW_KNOWLEDGE_BASE,
      tokenizer_name=EMBEDDING_MODEL_NAME
  )

  vector_store = Chroma.from_documents(
    docs_processed,
    embedding_model,
    persist_directory="db9",
    collection_metadata={"hnsw:space": "cosine"}
  )

  all_vector_stores.append(vector_store)

In [ ]:
len(all_vector_stores)

3

In [ ]:
prompt_in_chat_format = [
    {
        "role": "system",
        "content": """Using the information contained in the context, give a comprehensive answer to the question. Respond only to the question asked, response should be concise and relevant to the question. If the answer cannot be deduced from the context, do not give an answer.""",
    },
    {
        "role": "user",
        "content": """Context: {context}
        ---
        Now here is the question you need to answer.
        Question: {question}""",
    },
]

RAG_PROMPT_TEMPLATE = tokenizer.apply_chat_template(
    prompt_in_chat_format,
    tokenize=False, # Return a string, not token IDs
    add_generation_prompt=True # Ensures model knows where to start generating
)

print(RAG_PROMPT_TEMPLATE)

<|system|>
Using the information contained in the context, give a comprehensive answer to the question. Respond only to the question asked, response should be concise and relevant to the question. If the answer cannot be deduced from the context, do not give an answer.<|end|>
<|user|>
Context: {context}
        ---
        Now here is the question you need to answer.
        Question: {question}<|end|>
<|assistant|>



In [ ]:
answers_0=[]
for i in range(len(questions)):
  for j in range(len(all_vector_stores)):
    results = all_vector_stores[j].similarity_search_by_vector(
    embedding=embedding_model.embed_query(questions[i]), k=4
    )

    retrieved_docs_text = [doc.page_content for doc in results]
    context = "\nExtracted documents:\n"
    context += "".join([f"Document {str(i)}:::\n" + doc for i, doc in enumerate(retrieved_docs_text)])

    final_prompt = RAG_PROMPT_TEMPLATE.format(question=questions[i], context=context)

    answer = READER_LLM(final_prompt)[0]["generated_text"]
    answers_0.append(answer)

In [ ]:
len(answers)

9

In [ ]:
for i in range(len(answers)):
  print("---- answer {i} ----".format(i=i))
  answers=answers_0[i]
  print(answers_0[i])

---- answer 0 ----
 The new Pope in 2025 is Robert Prevost, known by the name Leo XIV. He was chosen during the first papal conclave following the death of Pope Francis on 21 April 2025.
---- answer 1 ----
 The new Pope in 2025 is Robert Prevost. He was announced as Pope Leo XIV following the first round of voting during the papal conclave that commenced on 7 May 2025.
---- answer 2 ----
 The new Pope in 2025 is Cardinal Robert Prevost, who takes the name Pope Leo XIV upon being elected during the first papal conclave following the death of Pope Francis I on 21 April 2025. He becomes the head of the Catholic Church and sovereign of the Vatican City State.
---- answer 3 ----
 The cardinals who did not participate in the 2025 Papal Conclave are Antonino Cañizares Llovera of Spain and John Njue of Kenya. Both declined participation either because they chose not to take part despite being able to, or health issues prevented them from attending. Additionally, Italian Cardinal Giovanni Angel

In [ ]:
reflections=["0 the answer is correct and adds some details",
             "1 the answer is correct but the details added are wrong (it's the fourth round of voting)",
             "2 the answer is correct and, as answer 0, added some details",
             "3 the answer identifies the 3 cardinals that did not participate, but the reasons are wrong and the spanish name is misspelled",
             "4 also this answer correctly identifies the 3 cardinals, the names are also misspelled or with more middle names, then, when stating the reasons it starts write",
             "5 this answer states correctly who did not participate and the reasons, but the spanish name is misspelled again",
             "6 the answer might seem ok at first glance, but the details are wrong and it starts to go out of track",
             "7 also this answer starts ok, but then it starts to lose the point",
             "8 as the previous answers this is similar"
]

### Template for your resulting json file with report

In [ ]:
your_results_dict = {
  "topic": "2025 papal conclave",
  "question":question_part0,
  "answer":answer_part0,
  "rag": [
    {
      "question": questions[0],
      "reason": reasons[0],
      "experiments": [
        {
          "chunk_size": chunk_sizes[0],
          "chunk_overlap": chunk_overlaps[0],
          "answer": answers[0],
          "reflection": reflections[0]
        },
        {
          "chunk_size": chunk_sizes[1],
          "chunk_overlap": chunk_overlaps[1],
          "answer": answers[1],
          "reflection": reflections[1]
        },
        {
          "chunk_size": chunk_sizes[2],
          "chunk_overlap": chunk_sizes[2],
          "answer": answers[2],
          "reflection": reflections[2]
        }
      ]
    },
    {
      "question": questions[1],
      "reason": reasons[1],
      "experiments": [
        {
          "chunk_size": chunk_sizes[0],
          "chunk_overlap": chunk_overlaps[0],
          "answer": answers[3],
          "reflection": reflections[3]
        },
        {
          "chunk_size": chunk_sizes[1],
          "chunk_overlap": chunk_overlaps[1],
          "answer": answers[4],
          "reflection": reflections[4]
        },
        {
          "chunk_size": chunk_sizes[2],
          "chunk_overlap": chunk_sizes[2],
          "answer": answers[5],
          "reflection": reflections[5]
        }
      ]
    },
    {
      "question": questions[2],
      "reason": reasons[2],
      "experiments": [
        {
          "chunk_size": chunk_sizes[0],
          "chunk_overlap": chunk_overlaps[0],
          "answer": answers[6],
          "reflection": reflections[6]
        },
        {
          "chunk_size": chunk_sizes[1],
          "chunk_overlap": chunk_overlaps[1],
          "answer": answers[7],
          "reflection": reflections[7]
        },
        {
          "chunk_size": chunk_sizes[2],
          "chunk_overlap": chunk_sizes[2],
          "answer": answers[8],
          "reflection": reflections[8]
        }
      ]
    }
  ]
}

In [ ]:
import json

with open("rag_report_Marta_Feltrin_2150892.json", "w", encoding="utf-8") as f:
    json.dump(your_results_dict, f, indent=2, ensure_ascii=False)